In [101]:
import requests
import pandas as pd
import os

# Corrected Overpass query for Spätis/corner stores
query = """
[out:json];
(
node[shop=convenience](52.4,13.2,52.7,13.6);
  node[shop=kiosk](52.4,13.2,52.7,13.6);
);
out;
"""

url = "https://overpass-api.de/api/interpreter"

response = requests.post(url, data=query)

if response.status_code == 200:
    try:
        data = response.json()
        elements = data.get("elements", [])
        df = pd.json_normalize(elements)

        # Save CSV to Documents
        folder = "/Users/harrisongoodman/Documents"
        os.makedirs(folder, exist_ok=True)
        file_path = os.path.join(folder, "spatis.csv")
        df.to_csv(file_path, index=False)
        print(f"CSV saved successfully to: {file_path}")

    except ValueError:
        print("Response is not in JSON format:")
        print(response.text[:200])
else:
    print(f"Request failed with status code: {response.status_code}")


CSV saved successfully to: /Users/harrisongoodman/Documents/spatis.csv


In [83]:
# Suppose your GeoDataFrame is called spatis_gdf
column_names = spatis_gdf.columns.to_list()
# Display all column names
print(column_names)




['geometry', 'addr:city', 'addr:country', 'addr:housenumber', 'addr:postcode', 'addr:street', 'addr:suburb', 'amenity', 'check_date:opening_hours', 'compressed_air', 'fuel:adblue', 'fuel:biodiesel', 'fuel:diesel', 'fuel:e10', 'fuel:octane_95', 'fuel:octane_98', 'name', 'opening_hours', 'operator', 'shop', 'wheelchair', 'brand', 'brand:wikidata', 'brand:wikipedia', 'fuel:GTL_diesel', 'fuel:biogas', 'fuel:cng', 'fuel:lpg', 'fuel:octane_102', 'surveillance', 'website', 'check_date', 'created_by', 'dog', 'email', 'fax', 'phone', 'start_date', 'indoor_seating', 'organic', 'outdoor_seating', 'smoking', 'opening_hours:signed', 'diet:halal', 'level', 'payment:credit_cards', 'payment:debit_cards', 'payment:apple_pay', 'payment:cards', 'payment:cash', 'payment:google_pay', 'payment:paypal', 'drink:club-mate', 'entrance', 'craft', 'post_office', 'post_office:brand', 'post_office:brand:wikidata', 'post_office:service_provider', 'name:signed', 'toilets:wheelchair', 'wheelchair:description', 'noname

In [84]:
# Explore all columns in the Spätis GeoDataFrame
spatis_gdf.describe(include="all").T


,count,unique,top,freq
geometry,151,151,POINT (13.2961178 52.4993218),1
addr:city,59,1,Berlin,59
addr:country,39,1,DE,39
addr:housenumber,82,59,14,4
addr:postcode,65,47,10827,3
...,...,...,...,...
room,0,0,NaN,NaN
height,0,0,NaN,NaN
building:part,0,0,NaN,NaN
name:ja,0,0,NaN,NaN


In [103]:
# Count missing values in each column
missing_count = spatis_gdf.isna().sum().sort_values(ascending=False)

# Show columns that have more than 200 missing values
print(missing_count[missing_count > 200])


Series([], dtype: int64)


In [86]:
# check unique values in 'barand' column
spatis_gdf['brand'].value_counts()

Series([], Name: count, dtype: int64)

In [104]:
#expland all columns to see more details
pd.set_option('display.max_columns', None)
print(spatis_gd.head(3))

                                name brand       operator  \
element id                                                  
node    63253672           Späti Joe   NaN            NaN   
        285096511  Späti am Comenius   NaN  H.Y. Soysüren   
        427670051              Späti   NaN            NaN   

                                addr:street addr:housenumber addr:postcode  \
element id                                                                   
node    63253672   Joachim-Friedrich-Straße               39         10711   
        285096511            Gubener Straße               43         10243   
        427670051                       NaN              NaN           NaN   

                      addr:suburb addr:city addr:country phone email website  \
element id                                                                     
node    63253672         Halensee    Berlin           DE   NaN   NaN     NaN   
        285096511  Friedrichshain    Berlin           DE   NaN 

In [105]:
# Selected Columns & Add Coordinates

In [89]:
import geopandas as gpd

# Example: Load your GeoDataFrame if not already loaded
# spatis_gdf = gpd.read_file("your_file.geojson")  # or .shp, etc.

# Ensure it's a GeoDataFrame
if not isinstance(spatis_gdf, gpd.GeoDataFrame):
    spatis_gdf = gpd.GeoDataFrame(spatis_gdf, geometry='geometry')

# Check the current CRS
print("Current CRS:", spatis_gdf.crs)

# Set CRS to WGS84 (lat/lon) if not already
spatis_gdf = spatis_gdf.to_crs(epsg=4326)

# Optional: Ensure geometry type is Point (needed for lat/lon extraction)
spatis_gdf = spatis_gdf[spatis_gdf.geometry.type == "Point"]

# Now you can extract lat/lon
spatis_gdf['latitude'] = spatis_gdf.geometry.y
spatis_gdf['longitude'] = spatis_gdf.geometry.x

# Quick check
print(spatis_gdf[['latitude', 'longitude']].head())


Current CRS: epsg:4326
                    latitude  longitude
element id                             
node    63253672   52.499322  13.296118
        285096511  52.511469  13.449035
        427670051  52.512027  13.496422
        427670057  52.512055  13.497071
        448594622  52.501098  13.441111


In [106]:
spatis_gdf['geometry'] = spatis_gdf['geometry'].apply(lambda geom: geom if geom.geom_type == 'Point' else geom.representative_point())
# Extract latitude and longitude
spatis_gdf["latitude"] = spatis_gdf.geometry.y
spatis_gdf["longitude"] = spatis_gdf.geometry.x
spatis_gdf


geometry addr:city addr:country  \
element id                                                              
node    63253672     POINT (13.29612 52.49932)    Berlin           DE   
        285096511    POINT (13.44904 52.51147)    Berlin           DE   
        427670051    POINT (13.49642 52.51203)       NaN          NaN   
        427670057    POINT (13.49707 52.51205)       NaN          NaN   
        448594622     POINT (13.44111 52.5011)       NaN          NaN   
...                                        ...       ...          ...   
        12793386255  POINT (13.47928 52.50199)    Berlin          NaN   
        12829967540  POINT (13.41345 52.54099)       NaN          NaN   
        13148808097  POINT (13.47074 52.50639)       NaN          NaN   
        13205182768  POINT (13.35372 52.49292)       NaN          NaN   
        13302705766  POINT (13.38302 52.49293)       NaN          NaN   

                    addr:housenumber addr:postcode               addr:street  \
element id                                                                     
node    63253672                  39         10711  Joachim-Friedrich-Straße   
        285096511                 43         10243            Gubener Straße   
        427670051                NaN           NaN                       NaN   
        427670057                243         10365         Frankfurter Allee   
        448594622                NaN           NaN                       NaN   
...                              ...           ...                       ...   
        12793386255               17         10317             Nöldnerstraße   
        12829967540                3           NaN           Danziger Straße   
        13148808097              NaN           NaN                       NaN   
        13205182768              NaN           NaN                       NaN   
        13302705766              NaN           NaN                       NaN   

                        addr:suburb amenity check_date:opening_hours  \
element id                                                             
node    63253672           Halensee     NaN                      NaN   
        285096511    Friedrichshain     NaN                      NaN   
        427670051               NaN     NaN                      NaN   
        427670057               NaN     NaN                      NaN   
        448594622               NaN     NaN               2024-03-29   
...                             ...     ...                      ...   
        12793386255             NaN     NaN                      NaN   
        12829967540             NaN     NaN                      NaN   
        13148808097             NaN     NaN                      NaN   
        13205182768             NaN     NaN                      NaN   
        13302705766             NaN     NaN                      NaN   

                    compressed_air fuel:adblue fuel:biodiesel fuel:diesel  \
element id                                                                  
node    63253672               NaN         NaN            NaN         NaN   
        285096511              NaN         NaN            NaN         NaN   
        427670051              NaN         NaN            NaN         NaN   
        427670057              NaN         NaN            NaN         NaN   
        448594622              NaN         NaN            NaN         NaN   
...                            ...         ...            ...         ...   
        12793386255            NaN         NaN            NaN         NaN   
        12829967540            NaN         NaN            NaN         NaN   
        13148808097            NaN         NaN            NaN         NaN   
        13205182768            NaN         NaN            NaN         NaN   
        13302705766            NaN         NaN            NaN         NaN   

                    fuel:e10 fuel:octane_95 fuel:octane_98  \
element id                                    

In [107]:
# Convert non-point geometries (e.g., polygons) into representative points
spatis_gdf['geometry'] = spatis_gdf['geometry'].apply(
    lambda geom: geom if geom.geom_type == 'Point' else geom.representative_point()
)

# Extract latitude and longitude
spatis_gdf["latitude"] = spatis_gdf.geometry.y
spatis_gdf["longitude"] = spatis_gdf.geometry.x

# Display the updated GeoDataFrame
spatis_gdf


geometry addr:city addr:country  \
element id                                                              
node    63253672     POINT (13.29612 52.49932)    Berlin           DE   
        285096511    POINT (13.44904 52.51147)    Berlin           DE   
        427670051    POINT (13.49642 52.51203)       NaN          NaN   
        427670057    POINT (13.49707 52.51205)       NaN          NaN   
        448594622     POINT (13.44111 52.5011)       NaN          NaN   
...                                        ...       ...          ...   
        12793386255  POINT (13.47928 52.50199)    Berlin          NaN   
        12829967540  POINT (13.41345 52.54099)       NaN          NaN   
        13148808097  POINT (13.47074 52.50639)       NaN          NaN   
        13205182768  POINT (13.35372 52.49292)       NaN          NaN   
        13302705766  POINT (13.38302 52.49293)       NaN          NaN   

                    addr:housenumber addr:postcode               addr:street  \
element id                                                                     
node    63253672                  39         10711  Joachim-Friedrich-Straße   
        285096511                 43         10243            Gubener Straße   
        427670051                NaN           NaN                       NaN   
        427670057                243         10365         Frankfurter Allee   
        448594622                NaN           NaN                       NaN   
...                              ...           ...                       ...   
        12793386255               17         10317             Nöldnerstraße   
        12829967540                3           NaN           Danziger Straße   
        13148808097              NaN           NaN                       NaN   
        13205182768              NaN           NaN                       NaN   
        13302705766              NaN           NaN                       NaN   

                        addr:suburb amenity check_date:opening_hours  \
element id                                                             
node    63253672           Halensee     NaN                      NaN   
        285096511    Friedrichshain     NaN                      NaN   
        427670051               NaN     NaN                      NaN   
        427670057               NaN     NaN                      NaN   
        448594622               NaN     NaN               2024-03-29   
...                             ...     ...                      ...   
        12793386255             NaN     NaN                      NaN   
        12829967540             NaN     NaN                      NaN   
        13148808097             NaN     NaN                      NaN   
        13205182768             NaN     NaN                      NaN   
        13302705766             NaN     NaN                      NaN   

                    compressed_air fuel:adblue fuel:biodiesel fuel:diesel  \
element id                                                                  
node    63253672               NaN         NaN            NaN         NaN   
        285096511              NaN         NaN            NaN         NaN   
        427670051              NaN         NaN            NaN         NaN   
        427670057              NaN         NaN            NaN         NaN   
        448594622              NaN         NaN            NaN         NaN   
...                            ...         ...            ...         ...   
        12793386255            NaN         NaN            NaN         NaN   
        12829967540            NaN         NaN            NaN         NaN   
        13148808097            NaN         NaN            NaN         NaN   
        13205182768            NaN         NaN            NaN         NaN   
        13302705766            NaN         NaN            NaN         NaN   

                    fuel:e10 fuel:octane_95 fuel:octane_98  \
element id                                    

In [91]:
# Select the 25 columns (fill missing with None if not present)

selected_columns = [
    #"osmid",
    "name", "brand", "operator",
    "addr:street", "addr:housenumber", "addr:postcode", "addr:suburb","addr:city", "addr:country",
    "phone", "email", "website", "opening_hours",
    "payment:visa", "payment:mastercard","payment:girocard", "dispensing", "delivery","surveillance","wheelchair", "building",
    "latitude", "longitude", "geometry",
    # placeholders for enrichment
    #"neighbourhood", "district",
    # add source info
    "source"
]  # Added the closing bracket here

In [92]:
# Rename map for only the columns that need renaming

rename_map = {
    "addr:street": "street",
    "addr:housenumber": "housenumber",
    "addr:postcode": "postcode",
    "addr:suburb": "suburb",
    "addr:city": "city",
    "addr:country": "country",
    "payment:visa": "payment_visa",
    "payment:mastercard": "payment_mastercard",
    "payment:girocard": "payment_girocard",
    "opening_hours": "openinghours",
    "wheelchair": "wheelchair_accessible",
    "building": "building_type"
}

In [93]:
# First, check what columns are actually available in the DataFrame
print(spatis_gdf.columns)

# Then select only the columns that exist
# Option 1: Remove 'dispensing' from your selected_columns list
selected_columns = [col for col in selected_columns if col != 'dispensing']
spatis_gd = spatis_gdf[selected_columns]

# Option 2: If you need 'dispensing' but it might have a different name
# Check if a similar column exists and use that instead
# For example, it might be called 'dispense' or 'dispensary' instead
# spatis_gd = spatis_gdf[corrected_column_list]

Index(['geometry', 'addr:city', 'addr:country', 'addr:housenumber',
       'addr:postcode', 'addr:street', 'addr:suburb', 'amenity',
       'check_date:opening_hours', 'compressed_air',
       ...
       'facebook', 'access', 'indoor', 'room', 'height', 'building:part',
       'name:ja', 'biergarten', 'latitude', 'longitude'],
      dtype='object', length=263)


In [94]:
# Preview the final DataFrame
spatis_gdf.head()

geometry addr:city addr:country  \
element id                                                            
node    63253672   POINT (13.29612 52.49932)    Berlin           DE   
        285096511  POINT (13.44904 52.51147)    Berlin           DE   
        427670051  POINT (13.49642 52.51203)       NaN          NaN   
        427670057  POINT (13.49707 52.51205)       NaN          NaN   
        448594622   POINT (13.44111 52.5011)       NaN          NaN   

                  addr:housenumber addr:postcode               addr:street  \
element id                                                                   
node    63253672                39         10711  Joachim-Friedrich-Straße   
        285096511               43         10243            Gubener Straße   
        427670051              NaN           NaN                       NaN   
        427670057              243         10365         Frankfurter Allee   
        448594622              NaN           NaN                       NaN   

                      addr:suburb amenity check_date:opening_hours  \
element id                                                           
node    63253672         Halensee     NaN                      NaN   
        285096511  Friedrichshain     NaN                      NaN   
        427670051             NaN     NaN                      NaN   
        427670057             NaN     NaN                      NaN   
        448594622             NaN     NaN               2024-03-29   

                  compressed_air fuel:adblue fuel:biodiesel fuel:diesel  \
element id                                                                
node    63253672             NaN         NaN            NaN         NaN   
        285096511            NaN         NaN            NaN         NaN   
        427670051            NaN         NaN            NaN         NaN   
        427670057            NaN         NaN            NaN         NaN   
        448594622            NaN         NaN            NaN         NaN   

                  fuel:e10 fuel:octane_95 fuel:octane_98  \
element id                                                 
node    63253672       NaN            NaN            NaN   
        285096511      NaN            NaN            NaN   
        427670051      NaN            NaN            NaN   
        427670057      NaN            NaN            NaN   
        448594622      NaN            NaN            NaN   

                                              name  \
element id                                           
node    63253672                         Späti Joe   
        285096511                Späti am Comenius   
        427670051                            Späti   
        427670057  Sheran Späti + Hermes Paketshop   
        448594622                 Späti am Schlesi   

                                                       opening_hours  \
element id                                                             
node    63253672                                                24/7   
        285096511  Mo-Th 08:00-00:00, Fr 08:00-01:00, Sa 08:00-01...   
        427670051                               PH,Mo-Su 06:00-24:00   
        427670057                                  Mo-Su 07:00-01:00   
        448594622                                                NaN   

                        operator         shop wheelchair brand brand:wikidata  \
element id                                                                      
node    63253672             NaN  convenience        NaN   NaN            NaN   
        285096511  H.Y. Soysüren        kiosk    limited   NaN            NaN   
        427670051            NaN  convenience        NaN   NaN            NaN   
        427670057            NaN  convenience         no   NaN            NaN   
        448594622            NaN  convenience         no   NaN            NaN   

                  brand:wikipedia fuel:GTL_diesel fuel:biogas fuel:cng  \
element id                 

In [95]:
# Data types and non-null counts

spatis_gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
MultiIndex: 151 entries, ('node', np.int64(63253672)) to ('node', np.int64(13302705766))
Columns: 263 entries, geometry to longitude
dtypes: float64(2), geometry(1), object(260)
memory usage: 389.0+ KB


In [96]:
import osmnx as ox
import geopandas as gpd

districts_gdf = ox.features_from_place(
    "Berlin, Germany",
    {"boundary": "administrative", "admin_level": "9"}
)

districts_gdf.head()


geometry  \
element  id                                                         
relation 16328  POLYGON ((13.30038 52.56998, 13.3019 52.57144,...   
         16330  POLYGON ((13.26423 52.62686, 13.26438 52.62802...   
         16334  POLYGON ((13.2082 52.59899, 13.20724 52.59987,...   
         16343  POLYGON ((13.10932 52.45071, 13.10956 52.45108...   
         16346  POLYGON ((13.27034 52.54934, 13.27061 52.54934...   

                      boundary                 name  ref admin_level name:ab  \
element  id                                                                    
relation 16328  administrative        Reinickendorf  NaN          10     NaN   
         16330  administrative              Frohnau  NaN          10     NaN   
         16334  administrative        Reinickendorf  NaN           9     NaN   
         16343  administrative              Spandau  NaN           9     NaN   
         16346  administrative  Charlottenburg-Nord  NaN          10     NaN   

               name:af name:als name:am name:an name:ang name:ar name:arc  \
element  id                                                                 
relation 16328     NaN      NaN     NaN     NaN      NaN     NaN      NaN   
         16330     NaN      NaN     NaN     NaN      NaN     NaN      NaN   
         16334     NaN      NaN     NaN     NaN      NaN     NaN      NaN   
         16343     NaN      NaN     NaN     NaN      NaN     NaN      NaN   
         16346     NaN      NaN     NaN     NaN      NaN     NaN      NaN   

               name:arz name:ast name:av name:az name:ba name:bar  \
element  id                                                         
relation 16328      NaN      NaN     NaN     NaN     NaN      NaN   
         16330      NaN      NaN     NaN     NaN     NaN      NaN   
         16334      NaN      NaN     NaN     NaN     NaN      NaN   
         16343      NaN      NaN     NaN     NaN     NaN      NaN   
         16346      NaN      NaN     NaN     NaN     NaN      NaN   

               name:bat-smg name:be name:be-tarask name:bg name:bi name:bn  \
element  id                                                                  
relation 16328          NaN     NaN            NaN     NaN     NaN     NaN   
         16330          NaN     NaN            NaN     NaN     NaN     NaN   
         16334          NaN     NaN            NaN     NaN     NaN     NaN   
         16343          NaN     NaN            NaN     NaN     NaN     NaN   
         16346          NaN     NaN            NaN     NaN     NaN     NaN   

               name:bo name:br name:bs name:bxr name:ca name:cbk-zam name:ce  \
element  id                                                                    
relation 16328     NaN     NaN     NaN      NaN     NaN          NaN     NaN   
         16330     NaN     NaN     NaN      NaN     NaN          NaN     NaN   
         16334     NaN     NaN     NaN      NaN     NaN          NaN     NaN   
         16343     NaN     NaN     NaN      NaN     NaN          NaN     NaN   
         16346     NaN     NaN     NaN      NaN     NaN          NaN     NaN   

               name:ckb name:co name:crh name:cs name:csb name:cu name:cv  \
element  id                                                                 
relation 16328      NaN     NaN      NaN     NaN      NaN     NaN     NaN   
         16330      NaN     NaN      NaN     NaN      NaN     NaN     NaN   
         16334      NaN     NaN      NaN     NaN      NaN     NaN     NaN   
         16343      NaN     NaN      NaN     NaN      NaN     NaN     NaN   
         16346      NaN     NaN      NaN     NaN      NaN     NaN     NaN   

               name:cy name:da        name:de name:diq name:dsb name:el  \
element  id                                                               
relation 16328     NaN     NaN  Reinickendorf      NaN      NaN     NaN   
         16330     NaN     NaN            NaN      NaN      NaN     NaN   
         16334     NaN     NaN          

In [97]:
districts_clean = districts_gdf[['geometry', 'name', 'admin_level']]
districts_clean.head()


geometry  \
element  id                                                         
relation 16328  POLYGON ((13.30038 52.56998, 13.3019 52.57144,...   
         16330  POLYGON ((13.26423 52.62686, 13.26438 52.62802...   
         16334  POLYGON ((13.2082 52.59899, 13.20724 52.59987,...   
         16343  POLYGON ((13.10932 52.45071, 13.10956 52.45108...   
         16346  POLYGON ((13.27034 52.54934, 13.27061 52.54934...   

                               name admin_level  
element  id                                      
relation 16328        Reinickendorf          10  
         16330              Frohnau          10  
         16334        Reinickendorf           9  
         16343              Spandau           9  
         16346  Charlottenburg-Nord          10

In [48]:
districts_clean['geometry'] = districts_clean['geometry'].buffer(0)


/opt/anaconda3/lib/python3.13/site-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [98]:
import os

# Create the directory structure if it doesn't exist
os.makedirs("spatis/sources", exist_ok=True)

# Now save the file
districts_clean.to_file("spatis/sources/berlin_districts.geojson", driver="GeoJSON")

In [59]:
spatis_gdf = spatis_gdf.to_crs("EPSG:4326")
districts_clean = districts_clean.to_crs("EPSG:4326")


In [60]:
spatis_with_districts = gpd.sjoin(
    spatis_gdf, 
    districts_clean, 
    how="left", 
    predicate="within"
)


In [61]:
spatis_with_districts = spatis_with_districts.rename(
    columns={
        "name_left": "store_name",
        "name_right": "district"
    }
)


In [99]:
spatis_with_districts.to_file("spatis/sources/spatis_with_districts.geojson", driver="GeoJSON")
